# Lab 9 – Parts D, E, F, G (Image-based version)

**▶ STEP 1:** Set `IMAGE_PATH` in the cell below to your image file path.  
**▶ STEP 2:** Run all cells top-to-bottom (`Kernel → Restart & Run All`).

Supported formats: `.jpg` / `.jpeg` / `.png` / `.bmp` / `.tif`

In [ ]:
# ══════════════════════════════════════════════════════════
# ▶▶▶  PUT YOUR IMAGE PATH HERE  ◀◀◀
IMAGE_PATH = "your_image.jpg"   # <── change this
# ══════════════════════════════════════════════════════════

## Imports & Setup

In [ ]:
import numpy as np
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import cv2
from skimage.feature import hog
from skimage import exposure
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

np.random.seed(42)

def show_image(img, title="Image", cmap="gray", figsize=(5, 5)):
    plt.figure(figsize=figsize)
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# ── Helper functions from Parts A/B/C ─────────────────────
def raw_moment(img, p, q):
    h, w = img.shape
    moment = 0.0
    for y in range(h):
        for x in range(w):
            moment += (x ** p) * (y ** q) * img[y, x]
    return moment

def centroid(img):
    m00 = raw_moment(img, 0, 0)
    if m00 == 0:
        raise ValueError("Cannot compute centroid: m00 is zero.")
    return raw_moment(img, 1, 0) / m00, raw_moment(img, 0, 1) / m00

def central_moment(img, p, q):
    h, w = img.shape
    cx, cy = centroid(img)
    mu = 0.0
    for y in range(h):
        for x in range(w):
            mu += ((x - cx) ** p) * ((y - cy) ** q) * img[y, x]
    return mu

def normalized_central_moment(img, p, q):
    mu_pq = central_moment(img, p, q)
    mu00  = central_moment(img, 0, 0)
    if mu00 == 0:
        raise ValueError("mu00 is zero.")
    gamma = 1 + (p + q) / 2
    return mu_pq / (mu00 ** gamma)

print("Setup complete.")

## Load & Pre-process Image

In [ ]:
_raw = cv2.imread(IMAGE_PATH)
if _raw is None:
    raise FileNotFoundError(
        f"\n\n  ✗  Could not open '{IMAGE_PATH}'.\n"
        "     Check that IMAGE_PATH is correct and the file exists.\n"
    )

gray_original  = cv2.cvtColor(_raw, cv2.COLOR_BGR2GRAY)
img_100        = cv2.resize(gray_original, (100, 100)).astype(np.float32)
img_64         = cv2.resize(gray_original, (64, 64)).astype(np.uint8)
img_hog_window = cv2.resize(gray_original, (64, 128)).astype(np.uint8)

print("Image loaded  :", IMAGE_PATH)
print("Original size :", gray_original.shape)
show_image(gray_original, "Loaded Image (grayscale)", figsize=(6, 6))

---
# Part D: Image Gradients

## Concept

The image gradient measures intensity change.

$$\nabla I = \left[\frac{\partial I}{\partial x},\; \frac{\partial I}{\partial y}\right]$$

Gradient magnitude: $G = \sqrt{G_x^2 + G_y^2}$

Gradient direction: $\theta = \tan^{-1}\!\left(\dfrac{G_y}{G_x}\right)$

## Task 12: Gradient Along X-Axis

In [ ]:
show_image(img_100, "Input Image (100×100)")

Gx = np.zeros_like(img_100)
Gx[:, :-1] = img_100[:, 1:] - img_100[:, :-1]

show_image(Gx, "Gradient Along X-axis")
print("Non-zero pixels in Gx:", np.count_nonzero(Gx))
print("Max value in Gx      :", Gx.max())

## Task 13: Gradient Along Y-Axis

In [ ]:
Gy = np.zeros_like(img_100)
Gy[:-1, :] = img_100[1:, :] - img_100[:-1, :]

show_image(Gy, "Gradient Along Y-axis")
print("Non-zero pixels in Gy:", np.count_nonzero(Gy))
print("Max value in Gy      :", Gy.max())

## Task 14: Gradient in Both Directions

In [ ]:
G_mag = np.sqrt(Gx ** 2 + Gy ** 2)
G_dir = np.arctan2(Gy, Gx)

show_image(Gx,    "Gx")
show_image(Gy,    "Gy")
show_image(G_mag, "Gradient Magnitude")

print("Gradient magnitude max :", G_mag.max())
print("Direction range (deg)  : [{:.1f}, {:.1f}]".format(
      np.degrees(G_dir.min()), np.degrees(G_dir.max())))

## Task 15: Forward, Backward, and Central Difference Filters

Forward:  $G_x(x,y) = I(x+1,y) - I(x,y)$

Backward: $G_x(x,y) = I(x,y) - I(x-1,y)$

Central:  $G_x(x,y) = \dfrac{I(x+1,y) - I(x-1,y)}{2}$

In [ ]:
def forward_difference_x(img):
    gx = np.zeros_like(img, dtype=np.float32)
    gx[:, :-1] = img[:, 1:] - img[:, :-1]
    return gx

def backward_difference_x(img):
    gx = np.zeros_like(img, dtype=np.float32)
    gx[:, 1:] = img[:, 1:] - img[:, :-1]
    return gx

def central_difference_x(img):
    gx = np.zeros_like(img, dtype=np.float32)
    gx[:, 1:-1] = (img[:, 2:] - img[:, :-2]) / 2
    return gx

gx_forward  = forward_difference_x(img_100)
gx_backward = backward_difference_x(img_100)
gx_central  = central_difference_x(img_100)

show_image(gx_forward,  "Forward Difference X")
show_image(gx_backward, "Backward Difference X")
show_image(gx_central,  "Central Difference X")

print("Forward  max:", gx_forward.max(),  "  non-zero:", np.count_nonzero(gx_forward))
print("Backward max:", gx_backward.max(), "  non-zero:", np.count_nonzero(gx_backward))
print("Central  max:", gx_central.max(),  "  non-zero:", np.count_nonzero(gx_central))

## Task 16: Sobel Filter

In [ ]:
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
sobel_y = np.array([[-1,-2,-1], [ 0, 0, 0], [ 1, 2, 1]], dtype=np.float32)

Gx_sobel   = cv2.filter2D(img_100, cv2.CV_32F, sobel_x)
Gy_sobel   = cv2.filter2D(img_100, cv2.CV_32F, sobel_y)
Gmag_sobel = np.sqrt(Gx_sobel ** 2 + Gy_sobel ** 2)

show_image(Gx_sobel,   "Sobel Gx")
show_image(Gy_sobel,   "Sobel Gy")
show_image(Gmag_sobel, "Sobel Gradient Magnitude")

print("Sobel Gx max  :", Gx_sobel.max())
print("Sobel Gy max  :", Gy_sobel.max())
print("Sobel Gmag max:", Gmag_sobel.max())

### Advanced Task: Gradient Stability Under Noise

In [ ]:
noise         = np.random.normal(0, 25, img_100.shape)
noisy_img     = np.clip(img_100 + noise, 0, 255).astype(np.float32)

gx_fwd_noisy  = forward_difference_x(noisy_img)
gx_cen_noisy  = central_difference_x(noisy_img)
gx_sob_noisy  = cv2.filter2D(noisy_img, cv2.CV_32F, sobel_x)

show_image(noisy_img,              "Noisy Image")
show_image(np.abs(gx_fwd_noisy),   "Forward Difference on Noisy Image")
show_image(np.abs(gx_cen_noisy),   "Central Difference on Noisy Image")
show_image(np.abs(gx_sob_noisy),   "Sobel X on Noisy Image")

print("Noise sensitivity (std of gradient response):")
print(f"  Forward difference : {np.abs(gx_fwd_noisy).std():.2f}")
print(f"  Central difference : {np.abs(gx_cen_noisy).std():.2f}")
print(f"  Sobel filter       : {np.abs(gx_sob_noisy).std():.2f}")

---
# Part E: Histograms

## Concept

A histogram counts how many pixels have each intensity value.
For an 8-bit grayscale image $I(x,y) \in [0,255]$, a full histogram has 256 bins.

> **Important limitation:** Histograms describe intensity distribution but ignore spatial arrangement.

## Task 17: Histogram of the Input Image

In [ ]:
img_uint8    = img_100.astype(np.uint8)
hist_input   = cv2.calcHist([img_uint8], [0], None, [256], [0, 256])

show_image(img_uint8, "Input Image (grayscale)")

plt.figure()
plt.plot(hist_input)
plt.title("Histogram of Input Image")
plt.xlabel("Intensity")
plt.ylabel("Pixel Count")
plt.tight_layout()
plt.show()

print("Total pixels          :", img_uint8.size)
print("Non-zero bins         :", np.count_nonzero(hist_input))
print("Dominant intensity bin:", int(np.argmax(hist_input)))

## Task 18: Histogram of Grayscale Image

In [ ]:
hist_gray = cv2.calcHist([img_uint8], [0], None, [256], [0, 256])

plt.figure()
plt.plot(hist_gray)
plt.title("Grayscale Histogram")
plt.xlabel("Intensity")
plt.ylabel("Pixel Count")
plt.tight_layout()
plt.show()

print("Total pixels              :", img_uint8.size)
print("Mean pixels per intensity :", float(hist_gray.mean()))

## Task 19: Normalized Histogram

In [ ]:
hist      = cv2.calcHist([img_uint8], [0], None, [256], [0, 256])
hist_norm = hist / hist.sum()

plt.figure()
plt.plot(hist_norm)
plt.title("Normalized Grayscale Histogram")
plt.xlabel("Intensity")
plt.ylabel("Probability")
plt.tight_layout()
plt.show()

print("Sum of normalized histogram =", round(float(hist_norm.sum()), 6))
print("Peak probability bin        =", int(np.argmax(hist_norm)))

## Task 20: Histograms Ignore Spatial Arrangement

In [ ]:
# Use top half vs bottom half to demonstrate spatial blindness
top_half    = img_uint8[:50, :]
bottom_half = img_uint8[50:, :]

hist_top    = cv2.calcHist([top_half],    [0], None, [256], [0, 256])
hist_bottom = cv2.calcHist([bottom_half], [0], None, [256], [0, 256])

show_image(top_half,    "Top Half of Image")
show_image(bottom_half, "Bottom Half of Image")

plt.figure()
plt.plot(hist_top,    label="Top Half")
plt.plot(hist_bottom, label="Bottom Half", linestyle="--")
plt.legend()
plt.title("Histogram Comparison – Top vs Bottom Half")
plt.xlabel("Intensity")
plt.ylabel("Pixel Count")
plt.tight_layout()
plt.show()

print("Histograms identical:", np.allclose(hist_top, hist_bottom))

---
# Part F: HOG — Histogram of Oriented Gradients

## Concept

HOG describes an image using local gradient directions.

```
Image → Gradients → Magnitude & Orientation → Cells
      → Orientation Histogram → Block Normalisation
      → HOG Descriptor → Recognition
```

## Task 21: Manual Understanding of HOG

In [ ]:
hog_input_f32 = img_64.astype(np.float32)

Gx_h = cv2.Sobel(hog_input_f32, cv2.CV_32F, 1, 0, ksize=3)
Gy_h = cv2.Sobel(hog_input_f32, cv2.CV_32F, 0, 1, ksize=3)

magnitude   = np.sqrt(Gx_h ** 2 + Gy_h ** 2)
orientation = np.degrees(np.arctan2(Gy_h, Gx_h)) % 180

show_image(img_64,      "Input Image (64×64)")
show_image(magnitude,   "Gradient Magnitude")
show_image(orientation, "Gradient Orientation")

print("Magnitude max    :", magnitude.max())
print("Orientation range: [{:.1f}, {:.1f}] degrees".format(
      orientation.min(), orientation.max()))

In [ ]:
# Orientation histogram for one 8×8 cell (top-left)
cell_mag  = magnitude[0:8, 0:8]
cell_ori  = orientation[0:8, 0:8]

bins      = 9
hist_cell = np.zeros(bins)
bin_width = 180 / bins

for i in range(cell_mag.shape[0]):
    for j in range(cell_mag.shape[1]):
        angle   = cell_ori[i, j]
        mag     = cell_mag[i, j]
        bin_idx = int(angle // bin_width)
        if bin_idx == bins:
            bin_idx = bins - 1
        hist_cell[bin_idx] += mag

print("HOG histogram for top-left 8×8 cell (9 orientation bins, 0–180°):")
for b, v in enumerate(hist_cell):
    print(f"  Bin {b} [{b*bin_width:.0f}–{(b+1)*bin_width:.0f} deg]: {v:.2f}")

## Task 22: HOG Descriptor Using scikit-image

In [ ]:
features, hog_image = hog(
    img_hog_window,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm="L2-Hys",
    visualize=True,
    feature_vector=True
)

hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))

show_image(img_hog_window,     "Input Image (64×128)")
show_image(hog_image_rescaled, "HOG Visualization")

print("HOG feature vector length:", len(features))

n_cells_y  = 128 // 8
n_cells_x  =  64 // 8
n_blocks_y = n_cells_y - 1
n_blocks_x = n_cells_x - 1
manual_len = n_blocks_y * n_blocks_x * 2 * 2 * 9
print(f"Manual check: {n_blocks_y}×{n_blocks_x} blocks × 4 cells × 9 bins = {manual_len}")

## Task 23: HOG-Based Recognition (Synthetic Shape Classifier)

In [ ]:
def create_shape(shape_type, size=64):
    img = np.zeros((size, size), dtype=np.uint8)
    if shape_type == "vertical_rectangle":
        cv2.rectangle(img, (25, 10), (39, 54), 255, -1)
    elif shape_type == "horizontal_rectangle":
        cv2.rectangle(img, (10, 25), (54, 39), 255, -1)
    elif shape_type == "circle":
        cv2.circle(img, (32, 32), 18, 255, -1)
    elif shape_type == "triangle":
        pts = np.array([[32, 10], [10, 54], [54, 54]], np.int32)
        cv2.fillPoly(img, [pts], 255)
    else:
        raise ValueError("Unknown shape type.")
    return img

def translate_image(img, tx, ty):
    h, w = img.shape
    M = np.float32([[1, 0, tx], [0, 1, ty]])
    return cv2.warpAffine(img, M, (w, h), borderValue=0)

shape_classes = ["vertical_rectangle", "horizontal_rectangle", "circle", "triangle"]

for shape_type in shape_classes:
    show_image(create_shape(shape_type), shape_type, figsize=(3, 3))

In [ ]:
X = []
y = []

for label, shape_type in enumerate(shape_classes):
    base_img = create_shape(shape_type)
    for i in range(30):
        tx = np.random.randint(-5, 6)
        ty = np.random.randint(-5, 6)
        sample   = translate_image(base_img, tx, ty)
        features = hog(
            sample,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm="L2-Hys",
            feature_vector=True
        )
        X.append(features)
        y.append(label)

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)
print("Labels  shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

clf = KNeighborsClassifier(n_neighbors=3)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, target_names=shape_classes))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

---
# Part G: Final Combined Recognition / Detection Pipeline

Complete recognition system using **Moment + Histogram + HOG** features.

| Feature | Dims |
|---|---|
| Histogram | 32 |
| HOG | 3780 |
| Moments | 7 |
| **Combined** | **3819** |

## Feature Extraction Functions

In [ ]:
def extract_histogram_features(img):
    hist = cv2.calcHist([img], [0], None, [32], [0, 256])
    hist = hist.flatten()
    hist = hist / (hist.sum() + 1e-8)
    return hist

def extract_hog_features(img):
    features = hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True
    )
    return features

def extract_moment_features(img):
    binary = (img > 127).astype(np.uint8)
    if binary.sum() == 0:
        return np.zeros(7, dtype=np.float64)
    features = []
    for p, q in [(2,0),(0,2),(1,1),(3,0),(0,3),(2,1),(1,2)]:
        features.append(normalized_central_moment(binary.astype(float), p, q))
    return np.array(features, dtype=np.float64)

def extract_combined_features(img):
    return np.concatenate([
        extract_histogram_features(img),   # 32 dims
        extract_hog_features(img),         # 3780 dims
        extract_moment_features(img)       # 7 dims
    ])                                     # Total: 3819

print("Feature dims -> histogram:32  hog:3780  moment:7  combined:3819")

## Data Augmentation Helper Functions

In [ ]:
def scale_shape_image(img, scale_factor):
    h, w = img.shape
    scaled = cv2.resize(img, None, fx=scale_factor, fy=scale_factor,
                        interpolation=cv2.INTER_NEAREST)
    canvas = np.zeros_like(img)
    sh, sw = scaled.shape
    crop_h = min(h, sh); crop_w = min(w, sw)
    y_c = (h  - crop_h) // 2; x_c = (w  - crop_w) // 2
    y_s = (sh - crop_h) // 2; x_s = (sw - crop_w) // 2
    canvas[y_c:y_c+crop_h, x_c:x_c+crop_w] = scaled[y_s:y_s+crop_h, x_s:x_s+crop_w]
    return canvas

def apply_brightness_change(img, factor):
    return np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)

def add_gaussian_noise(img, mean=0, std=10):
    noise = np.random.normal(mean, std, img.shape)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

def augment_shape_image(base_img):
    img = scale_shape_image(base_img, np.random.uniform(0.85, 1.15))
    img = translate_image(img, np.random.randint(-5, 6), np.random.randint(-5, 6))
    img = apply_brightness_change(img, np.random.uniform(0.8, 1.2))
    img = add_gaussian_noise(img, mean=0, std=10)
    return img

print("Augmentation: scale [0.85–1.15] + translate [-5,5]px + brightness [0.8–1.2] + noise σ=10")

## Final Classification Task

In [ ]:
X_combined = []
y_combined = []

for label, shape_type in enumerate(shape_classes):
    base_img = create_shape(shape_type)
    for i in range(40):
        sample   = augment_shape_image(base_img)
        features = extract_combined_features(sample)
        X_combined.append(features)
        y_combined.append(label)

X_combined = np.array(X_combined)
y_combined = np.array(y_combined)

print("Feature matrix shape:", X_combined.shape)
print("Label vector  shape :", y_combined.shape)

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_combined, y_combined, test_size=0.3, random_state=42, stratify=y_combined
)

clf_combined = KNeighborsClassifier(n_neighbors=3)
clf_combined.fit(X_train_c, y_train_c)

y_pred_c = clf_combined.predict(X_test_c)

print("=== Combined Feature Classifier ===")
print(classification_report(y_test_c, y_pred_c, target_names=shape_classes))
print("Confusion Matrix:")
print(confusion_matrix(y_test_c, y_pred_c))

## Optional Comparison: Each Feature Type Individually

In [ ]:
def build_dataset_with_feature_type(feature_type="hog", samples_per_class=30):
    X_data = []; y_data = []
    for label, shape_type in enumerate(shape_classes):
        base_img = create_shape(shape_type)
        for i in range(samples_per_class):
            sample = augment_shape_image(base_img)
            if   feature_type == "histogram": f = extract_histogram_features(sample)
            elif feature_type == "hog":       f = extract_hog_features(sample)
            elif feature_type == "moment":    f = extract_moment_features(sample)
            elif feature_type == "combined":  f = extract_combined_features(sample)
            else: raise ValueError("Unsupported feature type.")
            X_data.append(f); y_data.append(label)
    return np.array(X_data), np.array(y_data)

comparison_results = {}

for feature_type in ["histogram", "moment", "hog", "combined"]:
    print(f"\n{'='*55}")
    print(f"Feature type: {feature_type.upper()}")
    X_ft, y_ft = build_dataset_with_feature_type(feature_type, samples_per_class=30)
    X_tr_ft, X_te_ft, y_tr_ft, y_te_ft = train_test_split(
        X_ft, y_ft, test_size=0.3, random_state=42, stratify=y_ft)
    clf_ft = KNeighborsClassifier(n_neighbors=3)
    clf_ft.fit(X_tr_ft, y_tr_ft)
    y_pred_ft = clf_ft.predict(X_te_ft)
    acc = accuracy_score(y_te_ft, y_pred_ft)
    comparison_results[feature_type] = acc
    print(classification_report(y_te_ft, y_pred_ft, target_names=shape_classes))

print("\n" + "="*55)
print("SUMMARY")
print("-"*40)
for ft, acc in comparison_results.items():
    bar = "█" * int(acc * 20)
    print(f"  {ft:12s} : {acc:.3f}  {bar}")

---
## BONUS – Classify Your Uploaded Image

The classifier (trained on synthetic shapes) is now applied to your real image.

In [ ]:
img_for_predict = cv2.resize(gray_original, (64, 64))

show_image(img_for_predict, "Your Image (resized to 64×64 for prediction)")

feat_uploaded = extract_combined_features(img_for_predict)
pred_label    = clf_combined.predict([feat_uploaded])[0]

print(f"Predicted class : {shape_classes[pred_label]}")
print("(Classifier trained on synthetic shapes — for demonstration only.)")